# Coffee17 WR-PDR-HBP V1

Matched development experiment:

`WR-HBP`

vs

`WR-PDR-HBP = frozen WR-HBP logits + masked physical-descriptor residual`

The WR-HBP base is trained once per fold with strict deterministic CUDA settings,
then frozen. The 38-parameter physical residual is fitted on that fold's source/train
split only using deterministic CPU float64 L-BFGS-B.

No outer test is materialized.

Kaggle Input: **Coffee Green Bean with 17 Defects Original** only.

Output:

`wr-pdr-hbp-analysis-package.zip`


In [ ]:
NOTEBOOK_BUILD = "WR-PDR-HBP-V1"
SCIENTIFIC_CODE_COMMIT = "d8ca5ae424cd2e05bf5c7a6ff9d48214c0713ed7"

import importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

# Must be set before any CUDA context is initialized.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

print("NOTEBOOK_BUILD:", NOTEBOOK_BUILD)
print("SCIENTIFIC_CODE_COMMIT:", SCIENTIFIC_CODE_COMMIT)
print("CUBLAS_WORKSPACE_CONFIG:", os.environ["CUBLAS_WORKSPACE_CONFIG"])

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-wr-pdr-code"
PROJECT = WORK / "coffee17-wr-pdr-hbp-v1"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            env=os.environ.copy(),
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# Optional resume from an exact prior run of this project.
prior_projects = sorted(
    p for p in INPUT.rglob("coffee17-wr-pdr-hbp-v1")
    if p.is_dir()
)
if len(prior_projects) > 1:
    raise RuntimeError("Lebih dari satu prior WR-PDR project ditemukan.")
if prior_projects:
    print("MERGE PRIOR WR-PDR OUTPUT:", prior_projects[0])
    if PROJECT.exists():
        shutil.rmtree(PROJECT)
    shutil.copytree(prior_projects[0], PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 1. Exact scientific-code checkout.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)

run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run([
    "git", "-C", REPO, "checkout", "--quiet", "--detach",
    SCIENTIFIC_CODE_COMMIT
])

requirements = REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "pytest"])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

# Frozen unit tests.
run([
    sys.executable, "-m", "pytest", "-q",
    "tests/analysis/test_coffee17_physical_descriptors.py",
    "tests/modeling/test_physical_logit_residual.py",
])

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU sebelum Run All.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

CONFIG = REPO / "configs/physical_logit_residual_wr_hbp/WR_PDR_HBP_V1.yaml"
OUT = PROJECT / "experiments/coffee17-physical-logit-residual-wr-hbp-v1"
OUT.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Reconstruct exact Coffee17 provenance and folds.
# ------------------------------------------------------------
print("\n=== 1/4 RECONSTRUCT COFFEE17 ===")

by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(
        "Input harus Coffee17 original: "
        f"expected 979 images / 17 classes, observed {raw_count}/{len(by_class)}."
    )

ARCHIVE = WORK / "coffee17_wr_pdr_original.zip"
PROV = WORK / "coffee17_wr_pdr_provenance"
CANONICAL = WORK / "coffee17_wr_pdr_canonical"
FOLDS = WORK / "coffee17_wr_pdr_folds"

for path in (PROV, CANONICAL, FOLDS):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE,
    PROV,
    canonical_root=CANONICAL,
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV / "coffee17_provenance.json",
    FOLDS,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

print("clean_count:", fold_summary["clean_count"])
print("fold_counts:", json.dumps(fold_summary["fold_counts"], indent=2))

# ------------------------------------------------------------
# 3. Five fold-specific deterministic WR bases + physical residuals.
# ------------------------------------------------------------
print("\n=== 2/4 WR-HBP VS WR-PDR-HBP ===")

for fold in range(1, 6):
    print(f"\n================ FOLD {fold}/5 ================", flush=True)
    DEV = WORK / f"coffee17_wr_pdr_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)

    contract = materialize_preprocessing_development(
        CANONICAL,
        FOLDS / "clean_manifest.json",
        FOLDS / "fold_manifest.json",
        DEV,
        fold=fold,
    )
    assert contract["test_images_extracted"] is False
    assert not (DEV / "source/test").exists()

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_physical_logit_residual_wr_hbp",
        "--config", CONFIG,
        "--fold", str(fold),
        "--data-root", DEV,
        "--output-root", OUT,
        "--required-commit", SCIENTIFIC_CODE_COMMIT,
        "--device", "cuda:0",
        "--resume",
        "--authorize-training",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"WR_PDR_HBP_fold{fold}.log")

    result = json.loads(
        (OUT / f"fold_{fold}" / "seed42" / "pair_result.json").read_text(
            encoding="utf-8"
        )
    )
    d = result["DELTA_WR_PDR_MINUS_WR"]
    print(
        f"FOLD {fold} | "
        f"WR Macro={result['WR_HBP']['macro_f1']:.4f} | "
        f"PDR Macro={result['WR_PDR_HBP']['macro_f1']:.4f} | "
        f"Delta Macro={d['macro_f1']:+.4f} | "
        f"Delta Hard={d['hard_class_f1']:+.4f} | "
        f"Delta Worst={d['worst_class_f1']:+.4f} | "
        f"residual_params={result['active_residual_parameter_count']} | "
        f"QC train/val={result['train_descriptor_qc_fail_count']}/"
        f"{result['val_descriptor_qc_fail_count']}"
    )

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 4. Paired five-fold summary.
# ------------------------------------------------------------
print("\n=== 3/4 PAIRED SUMMARY ===")

SUMMARY = PROJECT / "analysis" / "wr_pdr_hbp_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_physical_logit_residual_wr_hbp_summary",
    "--output-root", OUT,
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
agg = summary["aggregate"]
delta = agg["DELTA_WR_PDR_MINUS_WR"]

print("\n=== KEY RESULT ===")
print("WR-HBP Macro-F1     :", f"{agg['WR_HBP']['macro_f1']['mean']:.4%}")
print("WR-PDR-HBP Macro-F1 :", f"{agg['WR_PDR_HBP']['macro_f1']['mean']:.4%}")
print("Delta Macro-F1      :", f"{delta['macro_f1']['mean']:+.4%}")
print("WR-HBP Hard-F1      :", f"{agg['WR_HBP']['hard_class_f1']['mean']:.4%}")
print("WR-PDR-HBP Hard-F1  :", f"{agg['WR_PDR_HBP']['hard_class_f1']['mean']:.4%}")
print("Delta Hard-F1       :", f"{delta['hard_class_f1']['mean']:+.4%}")
print("WR-HBP Worst-F1     :", f"{agg['WR_HBP']['worst_class_f1']['mean']:.4%}")
print("WR-PDR-HBP Worst-F1 :", f"{agg['WR_PDR_HBP']['worst_class_f1']['mean']:.4%}")
print("Delta Worst-F1      :", f"{delta['worst_class_f1']['mean']:+.4%}")
print("Positive Macro folds:", summary["positive_delta_folds"]["macro_f1"], "/ 5")
print("Rescue / damage      :",
      summary["paired_prediction_outcomes"]["rescue"], "/",
      summary["paired_prediction_outcomes"]["damage"])
print("Residual params      :", summary["active_residual_parameter_count"])
print("Strict determinism   :", summary["strict_determinism_verified_all_folds"])
print("SCREENING GATE       :", summary["screening_gate"]["decision"])

# ------------------------------------------------------------
# 5. Compact analysis package.
# ------------------------------------------------------------
print("\n=== 4/4 BUILD ANALYSIS PACKAGE ===")

PKG = WORK / "wr-pdr-hbp-analysis-package"
shutil.rmtree(PKG, ignore_errors=True)
PKG.mkdir(parents=True)

shutil.copytree(PROJECT / "analysis", PKG / "analysis")

for fold in range(1, 6):
    src_fold = OUT / f"fold_{fold}" / "seed42"
    dst_fold = PKG / f"fold_{fold}"
    dst_fold.mkdir(parents=True, exist_ok=True)

    shutil.copy2(src_fold / "pair_result.json", dst_fold / "pair_result.json")

    base_src = src_fold / "WR_HBP_BASE"
    base_dst = dst_fold / "WR_HBP_BASE"
    base_dst.mkdir(parents=True, exist_ok=True)
    for name in ("run_contract.json", "history.json", "run_config.yaml"):
        p = base_src / name
        if p.is_file():
            shutil.copy2(p, base_dst / name)
    if (base_src / "validation").is_dir():
        shutil.copytree(base_src / "validation", base_dst / "validation")

    residual_src = src_fold / "WR_PDR_HBP"
    residual_dst = dst_fold / "WR_PDR_HBP"
    residual_dst.mkdir(parents=True, exist_ok=True)
    for name in ("pair_result.json", "physical_residual_fit.json"):
        p = residual_src / name
        if p.is_file():
            shutil.copy2(p, residual_dst / name)
    for name in ("WR_HBP_validation", "WR_PDR_HBP_validation"):
        if (residual_src / name).is_dir():
            shutil.copytree(residual_src / name, residual_dst / name)

manifest_dir = PKG / "manifests"
manifest_dir.mkdir(parents=True, exist_ok=True)
for source in (
    PROV / "coffee17_provenance.json",
    PROV / "coffee17_raw_manifest.json",
    FOLDS / "clean_manifest.json",
    FOLDS / "fold_manifest.json",
    FOLDS / "fold_summary.json",
):
    shutil.copy2(source, manifest_dir / source.name)

shutil.copy2(CONFIG, PKG / "WR_PDR_HBP_V1.yaml")
protocol = REPO / "docs/protocols/PHYSICAL_LOGIT_RESIDUAL_WR_HBP_V1.md"
shutil.copy2(protocol, PKG / protocol.name)

metadata = {
    "notebook_build": NOTEBOOK_BUILD,
    "scientific_code_commit": SCIENTIFIC_CODE_COMMIT,
    "outer_test_materialized": False,
    "base_model": "WR_HBP",
    "candidate": "WR_PDR_HBP",
    "residual_fit": "CPU float64 L-BFGS-B",
    "active_residual_parameters": 38,
}
(PKG / "PACKAGE_METADATA.json").write_text(
    json.dumps(metadata, indent=2) + "\n",
    encoding="utf-8",
)

zip_path = Path(
    shutil.make_archive(
        str(WORK / "wr-pdr-hbp-analysis-package"),
        "zip",
        root_dir=PKG,
    )
)

print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Outer test was NOT materialized.")

for path in (REPO, PROV, CANONICAL, FOLDS, PKG):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
